In [1]:
import pandas as pd
import torch
import numpy as np
import napari
import zarr
from pathlib import Path
import tifffile
from src.data import _load_split_json

In [2]:
dfdata = pd.read_csv("./data/dfdata.csv")
split = _load_split_json("splits/binary_split_v1_ca_waves.json")

In [3]:
val_examples = ['Pax2_Calciumwaves_injectable anaesthetic19.0',
 'Pax2_Calciumwaves_injectable anaesthetic23.0',
 'Pax2_Calciumwaves_broken1.0',
 'Pax2_Calciumwave32.0',
 'Pax2_Calciumwaves_injectable anaesthetic11.0',
 'Pax2_Calciumwaves_ACP+isoflurane6.0',
 'Pax2_Calciumwave9.0',
 'Pax2_Calciumwaves_broken18.0',
 'Pax2_Calciumwaves_ACP+isoflurane7.0',
 'Pax2_Calciumwave45.0',
 'Pax2_Calciumwaves_broken17.0',
 'Pax2_Calciumwaves_ACP+isoflurane21.0',
 'Pax2_Calciumwave21.0',
 'Pax2_Calciumwave38.0',
 'Pax2_Calciumwaves_injectable anaesthetic5.0',
 'Pax2_Calciumwaves_ACP+isoflurane12.0',
 'Pax2_Calciumwaves_broken20.0',
 'Pax2_Calciumwaves_ACP+isoflurane4.0',
 'Pax2_Calciumwaves_injectable anaesthetic13.0',
 'Pax2_Calciumwaves_injectable anaesthetic1.0']

In [4]:
filename = Path(dfdata[dfdata['UniqueID'] == val_examples[1]]["image"].values[0].replace(r"/Volumes/Data","Z:"))
label_filename =Path(dfdata[dfdata['UniqueID'] == val_examples[1]]["label"].values[0].replace(r"/Volumes/Data","Z:"))
print(filename)

Z:\Users\Francesca\Data\2024\2024_11_01_Pax2-GCaMP6f_injectable anaesthesia\10\processedMovies\1-jumpCorrected.ome.zarr


In [5]:
movie = zarr.open(filename, mode="r")["s0"]
label = tifffile.imread(label_filename)

In [6]:
movie.shape

(5000, 480, 1024)

In [7]:
movie2 = movie[:]

In [8]:
from src import inference
from src.model import build_binary_model
from src.config import load_yaml_config
config = load_yaml_config("./config/binary_model_v2_ca_waves.yml")

model_config = config["model"]
validation_config = config["validation"]
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Cuda available")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

Cuda available


In [9]:
model = build_binary_model(
    architecture=model_config["architecture"],
    spatial_dims=model_config["spatial_dims"],
    in_channels=model_config["in_channels"],
    out_channels=model_config["out_channels"],
    channels=model_config["channels"],
    strides=model_config["strides"],
    num_res_units=model_config["num_res_units"],
).to(device)

checkpoint = torch.load("checkpoints/best_model_binary_v5_synaptic.pth")

model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [10]:
movie3 = torch.from_numpy(movie2)[None, None].to(device)


In [11]:
predictions = inference.predict_binary(model, movie3, device, validation_config["sw_batch_size"],validation_config["prediction_sigmoid"], 0.5, roi_size = [256,256,256])

52it [00:26,  1.97it/s]


Inference and assembly: 27.1s


In [12]:
import gc
import torch
del movie3
gc.collect()
torch.cuda.empty_cache()

In [14]:
app = napari.Viewer()

In [15]:
app.add_image(movie2)

<Image layer 'movie2' at 0x273ec234d70>

In [16]:
app.add_labels(label.astype(np.uint16))
app.add_labels(predictions.astype(np.uint16)*2,name='Prediction')

<Labels layer 'Prediction' at 0x2745d9a0410>

In [35]:
from monai.metrics import DiceMetric

In [47]:
dm = DiceMetric(
    reduction="mean_batch", return_with_label=True, num_classes=2
)
label_torch = (torch.from_numpy(label)[None,None]).to(device)
raw_scores = dm(predictions.to(device), label_torch)
print(dm.aggregate())

{'label_0': 0.9978, 'label_1': 0.0008}
